# AI-powered data-driven applications using pgvector, LangChain and LLMs

## Enviroment Setup

In [1]:
# Install dependencies.
!pip install asyncio==3.4.3 asyncpg==0.27.0 cloud-sql-python-connector["asyncpg"]==1.2.3
!pip install numpy==1.26.4 pandas
!pip install pgvector==0.1.8
!pip install langchain langchain_google_vertexai transformers
!pip install -U langchain-google-vertexai langchain-community
!pip install google-cloud-aiplatform
!pip install shapely
!pip install google-cloud-aiplatform[langchain,reasoningengine]
!pip install langchain-text-splitters
!pip install pydantic>=2.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 42.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [cloud-sql-python-connector]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 80.5 MB/s  0:00:006m0:00:01
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 98.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 22.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 104.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 99.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 791.8/791.8 kB 36.0 MB/s  0:00:00
  Attempting uninstall: ormsgpackm━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/22 [regex]nsors]
    Found existing installation: ormsgpack 1.7.0━━━━━━━━━━━━━━  4/22 [regex]
    Uninstalling ormsgpack-1.7.0:━━━━━━━━━━━━━━━━━━━━━━━━━━━━━  4/22 [r

In [2]:
# Automatically restart kernel after installation so that your environment can access the new packages.
import IPython

app = IPython.Application.instance()
app.kernel.do_shutdown(True)

{'status': 'ok', 'restart': True}

## install packages

In [1]:
import os
import pandas as pd
import vertexai
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel
from IPython.display import display, Markdown

from langchain_google_vertexai import VertexAIEmbeddings
import vertexai

project_id = "qwiklabs-gcp-02-c72a978ae543"
database_password = "cymbal@123!"
region =  "us-central1"
instance_name = "retail-ins"
database_name = "retail"
database_user = "retail-admin"

# Initialize Vertex AI SDK
vertexai.init(project=project_id, location=region)

/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1beta1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1beta1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.19) which Google will stop supporting in new releases of google.cloud.aiplatform_v1 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.aiplatform_v1 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/lib/python3.10/site-packages/google/cloud/aiplatform/models.py:52

## Create a Cloud SQL Instance and PostgreSQL database.

In [2]:
#@markdown Create and setup a Cloud SQL PostgreSQL instance, if not done already.
database_version = !gcloud sql instances describe {instance_name} --format="value(databaseVersion)"
if database_version[0].startswith("POSTGRES"):
  print("Found an existing Postgres Cloud SQL Instance!")
else:
  print("Creating new Cloud SQL instance...")
  !gcloud sql instances create {instance_name} --database-version=POSTGRES_15 \
  --region={region} --cpu=1 --memory=4GB --root-password={database_password}

# Create the database, if it does not exist.
out = !gcloud sql databases list --instance={instance_name} --filter="NAME:{database_name}" --format="value(NAME)"
if ''.join(out) == database_name:
  print("Database %s already exists, skipping creation." % database_name)
else:
  !gcloud sql databases create {database_name} --instance={instance_name}

# Create the database user for accessing the database.
!gcloud sql users create {database_user} \
--instance={instance_name} \
--password={database_password}

Creating new Cloud SQL instance...
Creating Cloud SQL instance for POSTGRES_15...done.                            
Created [https://sqladmin.googleapis.com/sql/v1beta4/projects/qwiklabs-gcp-02-c72a978ae543/instances/retail-ins].
NAME        DATABASE_VERSION  LOCATION       TIER              PRIMARY_ADDRESS  PRIVATE_ADDRESS  STATUS
retail-ins  POSTGRES_15       us-central1-c  db-custom-1-4096  35.222.113.71    -                RUNNABLE
Creating Cloud SQL database...done.                                            
Created database [retail].
instance: retail-ins
name: retail
project: qwiklabs-gcp-02-c72a978ae543
Creating Cloud SQL user...done.                                                
Created user [retail-admin].


In [ ]:
# @markdown Verify that you are able to connect to the database.

import asyncio
import asyncpg
from google.cloud.sql.connector import Connector


async def main():
    # get current running event loop to be used with Connector
    loop = asyncio.get_running_loop()
    # initialize Connector object as async context manager
    async with Connector(loop=loop) as connector:
        # create connection to Cloud SQL database
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  # Cloud SQL instance connection name
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}"
            # ... additional database driver args
        )

        # query Cloud SQL database
        results = await conn.fetch("SELECT version()")
        print(results[0]["version"])

        # close asyncpg connection
        await conn.close()


# Test connection with `asyncio`
await main() 

/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/__init__.py:23: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


PostgreSQL 15.15 on x86_64-pc-linux-gnu, compiled by Debian clang version 12.0.1, 64-bit


/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/instance.py:330: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  expiration = x509.not_valid_after


## Download and load the dataset in PostgreSQL

In [4]:
# Load dataset from a web URL and store it in a pandas dataframe.

import pandas as pd
import os

DATASET_URL = "https://github.com/GoogleCloudPlatform/python-docs-samples/raw/main/cloud-sql/postgres/pgvector/data/retail_toy_dataset.csv"
df = pd.read_csv(DATASET_URL)
df = df.loc[:, ["product_id", "product_name", "description", "list_price"]]
df = df.dropna()
df.head(10)

,product_id,product_name,description,list_price
0,7e8697b5b7cdb5a40daf54caf1435cd5,"Koplow Games Set of 2 D12 12-Sided Rock, Paper...","Rock, paper, scissors is a great way to resolv...",3.56
1,7de8b315b3cb91f3680eb5b88a20dcee,"12""-20"" Schwinn Training Wheels",Turn any small bicycle into an instrument for ...,28.17
2,fb9535c103d7d717f0414b2b111cfaaa,Bicycle Pinochle Jumbo Index Playing Cards - 1...,Purchase includes 1 blue deck and 1 red deck. ...,6.49
3,c73ea622b3be6a3ffa3b0b5490e4929e,Step2 Woodland Adventure Playhouse & Slide,The Step2 Woodland Climber Adventure Playhouse...,499.99
4,dec7bd1f983887650715c6fafaa5b593,Step2 Naturally Playful Welcome Home Playhouse...,Children can play and explore in the Step2 Nat...,600.00
5,74a695e3675efc2aad11ed73c46db29b,Slip N Slide Triple Racer with Slide Boogies,Triple Racer Slip and Slide with Boogie Boards...,37.21
6,3eae5293b56c25f63b47cb8a89fb4813,Hydro Tools Digital Pool/Spa Thermometer,The solar-powered Swimline Floating Digital Th...,15.92
7,ed85bf829a36c67042503ffd9b6ab475,Full Bucket Swing With Coated Chain Toddler Sw...,Safe Kids&Children Full Bucket Swing With Coa...,102.26
8,55820fa53f0583cb637d5cb2b051d78c,Banzai Water Park Splash Zone,Dive into fun in your own backyard with the B...,397.82
9,0e26a9e92e4036bfaa68eb2040a8ec97,Polaris 39-310 5-Liter Zippered Super Bag for ...,Keep your pool water sparkling clean all seaso...,39.47


In [ ]:
# Save the Pandas dataframe in a PostgreSQL table.

import asyncio
import asyncpg
from google.cloud.sql.connector import Connector


async def main():
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  # Cloud SQL instance connection name
            "asyncpg", 
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await conn.execute("DROP TABLE IF EXISTS products CASCADE")
        # Create the `products` table.
        await conn.execute(
            """CREATE TABLE products(
                                product_id VARCHAR(1024) PRIMARY KEY,
                                product_name TEXT,
                                description TEXT,
                                list_price NUMERIC)"""
        )

        # Copy the dataframe to the `products` table.
        tuples = list(df.itertuples(index=False))
        await conn.copy_records_to_table(
            "products", records=tuples, columns=list(df), timeout=10
        )
        await conn.close()


# Run the SQL commands now.
await main()  

/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/instance.py:330: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  expiration = x509.not_valid_after


## Generate vector embeddings using a Text Embedding model

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.schema import Document

from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[".", "\n"],
    chunk_size=500,
    chunk_overlap=0,
    length_function=len,
)

max_documents = 47  # Reduced limit to further control API usage
documents = []

# Create Document objects with product_id as metadata
for index, row in df.iterrows():
    product_id = row["product_id"]
    desc = row["description"]
    documents.append(Document(page_content=desc, metadata={"product_id": product_id}))

# Use the text splitter on a subset of documents (e.g., 40-50)
chunked = []
docs = text_splitter.split_documents(documents[40:max_documents])

# Collect split content along with product_id
for doc in docs:
    chunked.append({"product_id": doc.metadata["product_id"], "content": doc.page_content})

In [7]:
from langchain_google_vertexai import VertexAIEmbeddings
from google.cloud import aiplatform
import time

embeddings_service = VertexAIEmbeddings(model_name="text-embedding-005")

# Helper function to retry failed API requests with exponential backoff.
def retry_with_backoff(func, *args, retry_delay=10, backoff_factor=2.5, **kwargs):  # Increased delay and backoff factor
    max_attempts = 10
    retries = 0
    for i in range(max_attempts):
        try:
            return func(*args, **kwargs)
        except Exception as e:
            print(f"error: {e}")
            retries += 1
            wait = retry_delay * (backoff_factor**retries)
            print(f"Retry after waiting for {wait} seconds...")
            time.sleep(wait)

# Reduced batch size for API calls to manage quota limits
batch_size = 3
for i in range(0, len(chunked), batch_size):
    request = [x["content"] for x in chunked[i : i + batch_size]]
    response = retry_with_backoff(embeddings_service.embed_documents, request)
    # Store the retrieved vector embeddings for each chunk back.
    for x, e in zip(chunked[i : i + batch_size], response):
        x["embedding"] = e

# Store the generated embeddings in a pandas dataframe.
product_embeddings = pd.DataFrame(chunked)
product_embeddings.head()

/opt/conda/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


,product_id,content,embedding
0,8a6d71be41e01b284294ec488508b414,All of our productsWalmartply with internation...,"[0.04086191579699516, -0.020511694252490997, -..."
1,8a6d71be41e01b284294ec488508b414,. Holds Up to 6 Decks Fun for the whole family...,"[0.01160380244255066, -0.021363208070397377, -..."
2,9648838f5badebb9fc0b07f89cc29394,Better circulate water through your pool with ...,"[-0.005303527694195509, 0.017071831971406937, ..."
3,9648838f5badebb9fc0b07f89cc29394,".25-inch fitting (11070), 2 strainer grids (11...","[-0.01586345210671425, 0.017979448661208153, -..."
4,9648838f5badebb9fc0b07f89cc29394,. Circulate water through your pool with the h...,"[0.0184821505099535, 0.01372526679188013, 0.00..."


## Use pgvector to store the generated embeddings within PostgreSQL

In [ ]:
# Store the generated vector embeddings in a PostgreSQL table.

import asyncio
import asyncpg
from google.cloud.sql.connector import Connector
import numpy as np
from pgvector.asyncpg import register_vector


async def main():
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database.
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  # Cloud SQL instance connection name
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await conn.execute("CREATE EXTENSION IF NOT EXISTS vector")
        await register_vector(conn)

        await conn.execute("DROP TABLE IF EXISTS product_embeddings")
        # Create the `product_embeddings` table to store vector embeddings.
        await conn.execute(
            """CREATE TABLE product_embeddings(
                                product_id VARCHAR(1024) NOT NULL REFERENCES products(product_id),
                                content TEXT,
                                embedding vector(768))"""
        )

        # Store all the generated embeddings back into the database.
        for index, row in product_embeddings.iterrows():
            await conn.execute(
                "INSERT INTO product_embeddings (product_id, content, embedding) VALUES ($1, $2, $3)",
                row["product_id"],
                row["content"],
                np.array(row["embedding"]),
            )

        await conn.close()


# Run the SQL commands now.
await main()  # type: ignore

/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/instance.py:330: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  expiration = x509.not_valid_after


## Finding similar toy products using pgvector cosine search operator

In [ ]:
# @markdown Enter a short description of the toy to search for within a specified price range:
toy = "playing card games" 
min_price = 25  
max_price = 100  

# Quick input validations.
assert toy, "⚠️ Please input a valid input search text"

from langchain_google_vertexai import VertexAIEmbeddings
from google.cloud import aiplatform

aiplatform.init(project=f"{project_id}", location=f"{region}")

embeddings_service = VertexAIEmbeddings(model_name="text-embedding-005")
qe = embeddings_service.embed_query(toy)
from pgvector.asyncpg import register_vector
import asyncio
import asyncpg
from google.cloud.sql.connector import Connector

matches = []


async def main():
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database.
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await register_vector(conn)
        similarity_threshold = 0.1
        num_matches = 50

        # Find similar products to the query using the cosine similarity search
        # Over all vector embeddings. This new feature is provided by `pgvector`.
        results = await conn.fetch(
            """
                            WITH vector_matches AS (
                            SELECT product_id, 1 - (embedding <=> $1) AS similarity
                            FROM product_embeddings
                            WHERE 1 - (embedding <=> $1) > $2
                            ORDER BY similarity DESC
                            LIMIT $3
                            )
                            SELECT product_name, list_price, description FROM products
                            WHERE product_id IN (SELECT product_id FROM vector_matches)
                            AND list_price >= $4 AND list_price <= $5
                            """,
            qe,
            similarity_threshold,
            num_matches,
            min_price,
            max_price,
        )

        if len(results) == 0:
            raise Exception("Did not find any results. Adjust the query parameters.")

        for r in results:
            # Collect the description for all the matched similar toy products.
            matches.append(
                {
                    "product_name": r["product_name"],
                    "description": r["description"],
                    "list_price": round(r["list_price"], 2),
                }
            )

        await conn.close()


# Run the SQL commands now.
await main()  

# Show the results for similar products that matched the user query.
matches = pd.DataFrame(matches)
matches.head(5)

/opt/conda/lib/python3.10/site-packages/vertexai/_model_garden/_model_garden_models.py:278: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


,product_name,description,list_price
0,Intex 26004E Above Ground Swimming Pool Inlet ...,Better circulate water through your pool with ...,39.99
1,KARMAS PRODUCT Heavy Duty Glider Swing for Kid...,KARMAS PRODUCT Heavy Duty Glider Swing for Kid...,49.99
2,Melissa & Doug Jumbo ABC-123 Rug (58 x 79 inch...,Kids will have jumbo amounts of fun exploring ...,54.99
3,Champion Sports Tournament Series Horseshoe Set,The Champion Sports Deluxe Horseshoe Tournamen...,61.06


## LLMs and LangChain

### Building an AI-curated contextual hybrid search

In [ ]:
# @markdown Enter the user search query in a simple English text. The price filters are shown separately here for demo purposes. These filters may represent additional input from your frontend application.

user_query = "Do you have a beach toy set that teaches numbers and letters to kids?"  
min_price = 20  
max_price = 100  

# Quick input validations.
assert user_query, "⚠️ Please input a valid input search text"

In [11]:
qe = embeddings_service.embed_query(user_query)

In [ ]:
from pgvector.asyncpg import register_vector
import asyncio
import asyncpg
from google.cloud.sql.connector import Connector

matches = []


async def main():
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database.
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await register_vector(conn)
        similarity_threshold = 0.5
        num_matches = 10

        # Find similar products to the query using cosine similarity search
        # over all vector embeddings. This new feature is provided by `pgvector`.
        results = await conn.fetch(
            """
                            WITH vector_matches AS (
                            SELECT product_id, 1 - (embedding <=> $1) AS similarity
                            FROM product_embeddings
                            WHERE 1 - (embedding <=> $1) > $2
                            ORDER BY similarity DESC
                            LIMIT $3
                            )
                            SELECT product_name, list_price, description FROM products
                            WHERE product_id IN (SELECT product_id FROM vector_matches)
                            AND list_price >= $4 AND list_price <= $5
                            """,
            qe,
            similarity_threshold,
            num_matches,
            min_price,
            max_price,
        )
        if len(results) == 0:
            raise Exception("Did not find any results. Adjust the query parameters.")

        for r in results:
            # Collect the description for all the matched similar toy products.
            matches.append(
                f"""The name of the toy is {r["product_name"]}.
                        The price of the toy is ${round(r["list_price"], 2)}.
                        Its description is below:
                        {r["description"]}."""
            )
        await conn.close()


# Run the SQL commands now.
await main()  

# Show the results for similar products that matched the user query.
matches

/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/instance.py:330: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  expiration = x509.not_valid_after


['The name of the toy is Champion Sports Tournament Series Horseshoe Set.\n                        The price of the toy is $61.06.\n                        Its description is below:\n                        The Champion Sports Deluxe Horseshoe Tournament Set features everything you need to set up a horseshoe tournament, including a set of rules to ensure games stay on track. The tough steel horseshoes are plated with chrome and brass for an attractive shine and extra durability, and the set includes two rugged chrome-plated steel stakes to take aim at. The horseshoe set comes with a carrying bag made from strong 1680 denier nylon fabric that can endure toting the sturdy heavy metal pieces around with ease. Two bronze powder coated steel horseshoes Two silver powder coated steel horseshoes Two silver powder coated solid steel stakes One durable weather-resistant 1680 Denier nylon carry bag with YKK zipper and embroidered logo Two bronze powder coated steel horseshoesTwo silver powder co

In [ ]:
# Using LangChain for summarization and efficient context building.

from langchain.chains.summarize import load_summarize_chain
from langchain.docstore.document import Document
from langchain_google_vertexai import VertexAI
from langchain_core.prompts import PromptTemplate
from IPython.display import display, Markdown

llm = VertexAI(temperature=0.7)

map_prompt_template = """
            You will be given a detailed description of a toy product.
            This description is enclosed in triple backticks (```).
            Using this description only, extract the name of the toy,
            the price of the toy and its features.

            ```{text}```
            SUMMARY:
            """
map_prompt = PromptTemplate(template=map_prompt_template, input_variables=["text"])

combine_prompt_template = """
                You will be given a detailed description different toy products
                enclosed in triple backticks (```) and a question enclosed in
                double backticks(``).
                Select one toy that is most relevant to answer the question.
                Using that selected toy description, answer the following
                question in as much detail as possible.
                You should only use the information in the description.
                Your answer should include the name of the toy, the price of the toy
                and its features. Your answer should be less than 200 words.
                Your answer should be in Markdown in a numbered list format.


                Description:
                ```{text}```


                Question:
                ``{user_query}``


                Answer:
                """
combine_prompt = PromptTemplate(
    template=combine_prompt_template, input_variables=["text", "user_query"]
)

docs = [Document(page_content=t) for t in matches]
chain = load_summarize_chain(
    llm, chain_type="map_reduce", map_prompt=map_prompt, combine_prompt=combine_prompt
)
output = chain.invoke(
    {
        "input_documents": docs,
        "user_query": user_query,
    }
)


# Extract and display the output
answer = output.get('output_text', ' ')
display(Markdown(answer))

The most relevant toy to the question is the **Melissa & Doug Jumbo ABC-123 Rug**.

Here's why and the requested information:

1.  The Melissa & Doug Jumbo ABC-123 Rug is priced at \$54.99.
2.  It features an illustrated alphabet, numbers 1-9, shapes, and colors.
3.  It includes 36 double-sided game cards.
4.  The rug is made of soft, durable material and has a skid-proof backing and reinforced border binding.


### Adding AI-powered creative content generation

In [ ]:
# @markdown Describe your a new product in just a few words:

creative_prompt = "A bicycle with brand name 'Roadstar bike' for kids that comes with training wheels and helmet." 

# Quick input validations.
assert creative_prompt, "⚠️ Please input a valid input search text"

In [ ]:
from pgvector.asyncpg import register_vector
import asyncio
import asyncpg
from google.cloud.sql.connector import Connector

qe = embeddings_service.embed_query(creative_prompt)
qe_str = "[%s]" % (",".join([str(x) for x in qe]))
matches = []


async def main():
    loop = asyncio.get_running_loop()
    async with Connector(loop=loop) as connector:
        # Create connection to Cloud SQL database.
        conn: asyncpg.Connection = await connector.connect_async(
            f"{project_id}:{region}:{instance_name}",  
            "asyncpg",
            user=f"{database_user}",
            password=f"{database_password}",
            db=f"{database_name}",
        )

        await register_vector(conn)
        similarity_threshold = 0.5

        # Find similar products to the query using cosine similarity search
        # over all vector embeddings. This new feature is provided by `pgvector`.
        results = await conn.fetch(
            """
                            WITH vector_matches AS (
                            SELECT product_id, 1 - (embedding <=> $1) AS similarity
                            FROM product_embeddings
                            WHERE 1 - (embedding <=> $2) > $3
                            ORDER BY similarity DESC
                            LIMIT 1
                            )
                            SELECT description FROM products
                            WHERE product_id IN (SELECT product_id FROM vector_matches)
                            """,
            qe,
            qe,
            similarity_threshold,
        )

        for r in results:
            matches.append(r["description"])

        await conn.close()


# Run the SQL commands now.
await main()  

# Show the matched product description.
matches

/opt/conda/lib/python3.10/site-packages/google/cloud/sql/connector/instance.py:330: CryptographyDeprecationWarning: Properties that return a naïve datetime object have been deprecated. Please switch to not_valid_after_utc.
  expiration = x509.not_valid_after


['KARMAS PRODUCT Heavy Duty Glider Swing for Kids Fun Swing Seat; Heavy Duty Glider Swing,play more fun! Use highly durable plastic ,very strong. Features footrests and handholds for added comfort and security Push with your feet; pull with your hands to get this glider swing flying Meets US ASTM safety standards Suitable for children 3 years up,Max weight is 55 pounds(25KG) Use highly durable plastic ,very strong.Features footrests and handholds for added comfort and securityPush with your feet; pull with your hands to get this glider swing flyingMeets US ASTM safety standardsSuitable for children 3 years up,Max weight is 55 pounds(25KG)']

In [ ]:
from IPython.display import display, Markdown
from langchain_google_vertexai import VertexAI
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableSequence

template = """
            You are given descriptions about some similar kind of toys in the context.
            This context is enclosed in triple backticks (```).
            Combine these descriptions and adapt them to match the specifications in
            the initial prompt. All the information from the initial prompt must
            be included. You are allowed to be as creative as possible,
            and describe the new toy in as much detail. Your answer should be
            less than 200 words.

            Context:
            ```{context}```

            Initial Prompt:
            {creative_prompt}
            Answer:
        """

prompt = PromptTemplate(
    template=template, input_variables=["context", "creative_prompt"]
)


# Define the LLM
llm = VertexAI(temperature=0.7)
# Example `matches` list

matches = [
    {"description": "This is a toy description 1."},
    {"description": "This is a toy description 2."},
    {},  # Missing `description`
    "Invalid item" 
]

# Construct the context by extracting valid descriptions
context = "\n".join(
    item["description"] for item in matches if isinstance(item, dict) and "description" in item
)

# Use RunnableSequence for chaining
llm_chain = RunnableSequence(prompt | llm)
# Invoke the chain
answer = llm_chain.invoke({
    "context": context,
    "creative_prompt": creative_prompt,
})

# Display the answer in Markdown format
display(Markdown(answer))

```This is a toy description 1. The 'Zoomster' balance bike helps toddlers develop coordination and balance before transitioning to a pedal bike. It features a lightweight frame, adjustable seat, and puncture-proof tires for safe and comfortable riding. Available in vibrant colors.

This is a toy description 2. The 'Trailblazer' kids' mountain bike is perfect for adventurous youngsters. It boasts front suspension, reliable brakes, and multiple gears for tackling varied terrain. Safety features include reflectors and a chain guard. Durable and built to last.```

Introducing the 'Roadstar Bike' – the perfect first bicycle for your child! This vibrant and sturdy bike comes equipped with removable training wheels, making it ideal for kids learning to ride. Safety is paramount, so each 'Roadstar Bike' includes a matching helmet. Inspired by balance bikes, the lightweight frame and adjustable seat ensure a comfortable fit. Built to last like a mountain bike, the 'Roadstar Bike' will provide years of fun and adventure. Get your little one rolling with confidence!
